# Capability Harvesting

## Preparation

In [ ]:
import json
import re

from typing import Optional
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif

In [ ]:
# pd.set_option('display.max_rows', None)

In [ ]:
def prepare_dataset(datasets):
    tidy_dfs = []
    
    for dataset in datasets:
        wide_df = (
            pd
                .read_csv(dataset)
                .set_index("Capability")
                .T
                .reset_index(names="endpoint")
        )
    
        tidy_df = wide_df.melt(
            id_vars="endpoint",
            var_name="name",
            value_name="value"
        )
    
        tidy_dfs.append(tidy_df)
    
    df = pd.concat(tidy_dfs, ignore_index=True)
    df = df.dropna(subset=["value"])
    df = df.drop_duplicates(subset=["endpoint", "name"])
    df[["handle", "device_id"]] = (
        df["endpoint"]
            .copy()
            .str.split("#", n=1, expand=True)
    )
    df["name"] = df["name"].str.removeprefix("client-data.")

    return df

In [ ]:
dataset_our = ["ours_2025-12-17_01.csv"]

df = prepare_dataset(dataset_our)

In [ ]:
df

## Analysis on harvested devices

In [ ]:
df["device_id"].unique()

In [ ]:
df.groupby("handle")["device_id"].nunique().agg(
    min_devices="min",
    max_devices="max",
    avg_devices="mean",
)

In [ ]:
df.groupby("handle")["device_id"].nunique().value_counts()

In [ ]:
(
    df
        .groupby(["handle", "device_id"])["name"]
        .nunique()
        .rename("cap_count")
        .hist()
)

In [ ]:
n_endpoints_total = (
    df[["handle", "device_id"]]
        .drop_duplicates()
        .shape[0]
)

In [ ]:
cap_count = (
    df
        .drop_duplicates(["handle", "device_id", "name"])
        .groupby("name")
        .size()
        .rename("n_endpoints")
        .reset_index()
        .sort_values("n_endpoints", ascending=False)
)
cap_count

In [ ]:
cap_count.plot(kind="bar")

In [ ]:
cap_count["share"] = cap_count["n_endpoints"] / n_endpoints_total
# cap_count.to_csv("cap_count.csv")
cap_count

In [ ]:
def classify_n_endpoints(n):
    if n == n_endpoints_total:
        return "universal"
    elif n > 0.5 * n_endpoints_total:
        return "common"
    elif n > 10:
        return "rare"
    else:
        return "very_rare"

cap_count["class"] = (cap_count["n_endpoints"]
    .apply(classify_n_endpoints))

In [ ]:
cap_count

## Import First Seen / Last Seen

In [ ]:
df_fs = pd.read_csv("first_last_seen.csv")
df_fs = df_fs[["device_type", "name", "first_os_version"]]

In [ ]:
def parse_os_version(v):
    """
    '26.2' -> (26, 2)
    """
    return tuple(int(p) for p in v.split("."))

def device_type_from_model(model: str):
    if model.startswith("iPhone"):
        return "iPhone"
    if model.startswith("iPad"):
        return "iPad"
    if model.startswith("Watch"):
        return "Watch"
    if "Mac" in model:
        return "Mac"
    if "Vision" in model:
        return "VisionPro"
    raise ValueError(f"Unknown Device Type from model: {model}")

df_fs["first_os_version_key"] = df_fs["first_os_version"].apply(parse_os_version)

In [ ]:
df_fs

In [ ]:
def infer_endpoint_version(group):
    return (
        group[["name"]]
            .dropna()
            .drop_duplicates()
            .merge(
                df_fs[["name", "device_type", "first_os_version_key"]],
                on="name",
                how="left"
            )
            .groupby("device_type", as_index=False)["first_os_version_key"]
            .max()
            .set_index("device_type")["first_os_version_key"]
            .rename(lambda d: f"os_candidate_{d}")
            .to_dict()
    )

In [ ]:
version_classifier = (
    df
        .drop_duplicates(["handle", "device_id", "name"])
        .groupby(["handle", "device_id"], as_index=False)
        .apply(lambda g: pd.Series(infer_endpoint_version(g)), include_groups=False)
)
version_classifier

## Classifiers

In [ ]:
all_caps_versions = pd.read_csv("all_caps_version.csv")
all_caps_versions["typed_value"] = all_caps_versions["typed_value"].replace(
    {"True": True, "False": False}
)
all_caps_versions

In [ ]:
df["typed_value"] = df["value"].replace(
    {"T": True, "F": False}
)

## Device Classification

In [ ]:
dev_df = pd.read_csv("all_caps_devices.csv")
dev_df["os_version"] = dev_df["os_version"].apply(parse_os_version)
dev_df["typed_value"] = dev_df["typed_value"].replace({
    "True": True, "False": False, "true": True, "false": False
})
dev_df["source"] = "static"
dev_df["verified"] = False

In [ ]:
def dev_caps_ge_os(dev_df, lower_bound: tuple, device_type: Optional[str]):
    mask = dev_df["os_version"] >= lower_bound

    if device_type:
        mask = mask & (dev_df["device_type"] == device_type)
    
    return dev_df[mask]

In [ ]:
dev_caps_ge_os(dev_df, (26,0), None)

In [ ]:
def dev_caps_for_device(dev_df, device_model: str, os_version: tuple):
    mask = (dev_df["os_version"] == os_version) & (dev_df["device_model"] == device_model)
    
    return dev_df[mask]

In [ ]:
dev_caps_for_device(dev_df, "iPhone14,7", (26,3)).query("name == 'supports-heif'")["typed_value"].iloc[0]

In [ ]:
def label_cap(row):
    name = row["name"]
    value = row["typed_value"]
    
    if pd.isna(value):
        return f"VALUE::{name}::NA"
        
    if value == False:
        return f"VALUE::{name}::false"
    else:
        return f"VALUE::{name}::{value}"

def refresh_labels(df):
    df["label"] = df.apply(label_cap, axis=1)
    return df

In [ ]:
refresh_labels(dev_df)

In [ ]:
def get_caps_for_probe(handle: str, device_id: str):
    mask = (df["handle"] == handle) & (df["device_id"] == device_id)
    return df[mask][["handle", "device_id", "name", "value", "typed_value"]]

In [ ]:
def remove_ignored_caps(df):
    return df[df["name"] != "session-token-refresh-seconds"]

In [ ]:
def learn_v1(df_dev_caps, df_probe_caps, device_model, os_version):
    df_dev_caps_row = dev_caps_for_device(df_dev_caps, device_model, os_version)

    
    to_create = []

    for idx, cap in df_probe_caps.iterrows():
        # print(cap["name"], cap["typed_value"])

        if cap["name"] in ("session-token-refresh-seconds",):
            continue

        row = df_dev_caps_row[df_dev_caps_row["name"] == cap["name"]]

        probe_values = {
            "typed_value": cap["typed_value"],
            "verified": True
        }

        
        if row.shape[0] > 0:
            # existing cap entry
            entry = row.iloc[0]

            # Conflicts check
            new_val = cap["typed_value"]
            existing_val = entry["typed_value"]
            if (new_val != existing_val) and (entry["source"] == "static") and not (pd.isna(existing_val)):
                print(
                    f"Warning! Overriding static value for {cap["name"]}:",
                    f"static={entry["typed_value"]}, probe={cap["typed_value"]}"
                )
                probe_values["source"] = "probe"

            # Update
            df_dev_caps.loc[row.index[0], list(probe_values.keys())] = list(probe_values.values())
        else:
            print(f"Info: No existing entry for {cap["name"]}, creating...")
            to_create.append({
                "device_model": device_model,
                "os_version": os_version,
                "name": cap["name"],
                "source": "probe"} | probe_values)

    result = df_dev_caps
    if to_create:
        pd_new = pd.DataFrame(to_create)
        pd_new["device_type"] = pd_new["device_model"].map(device_type_from_model)
        result = pd.concat([df_dev_caps, pd_new], ignore_index=True)

    return result

In [ ]:
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "0"), "UniversalMac", (15, 7, 3))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "1"), "iPad_Spring_2022", (26, 2))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "2"), "Watch7,1", (26, 2))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "3"), "iPhone17,1", (26, 2))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "0"), "iPhone18,1", (26, 2))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "1"), "Watch8,1", (26, 1))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "2"), "UniversalMac", (26, 2))
dev_df = learn_v1(dev_df, get_caps_for_probe("<<REDACTED>>", "0"), "iPhone17,1", (26, 2))
refresh_labels(dev_df)

In [ ]:
dev_caps_for_device(dev_df, "iPhone17,1", (26,2)).head()#.to_csv("iPhone17,1_26,2.csv")

In [ ]:
verified_caps = dev_df[dev_df["verified"] == True].reset_index(drop=True)
# verified_caps = dev_df[(dev_df["verified"] == True) | ((dev_df["verified"] == False) & (~dev_df["typed_value"].isna()))].copy().reset_index(drop=True)
# verified_caps = dev_df[~dev_df["typed_value"].isna()].copy().reset_index(drop=True)
# verified_caps = dev_df[dev_df["os_version"] >= (26,0)].copy().reset_index(drop=True)
verified_caps

In [ ]:
caps_pivot = (
    verified_caps
        .pivot_table(
            index=["device_type", "device_model", "os_version"],
            columns="name",
            values="typed_value",
            aggfunc="first"
        )
)
caps_pivot

In [ ]:
X_raw = caps_pivot.copy()
X_raw = X_raw.dropna(axis=1, how="all")
X_str = X_raw.astype("string").fillna("<<MISSING>>")
X_enc = X_str.apply(lambda s: pd.factorize(s, sort=True)[0]).to_numpy()

y = caps_pivot.index.get_level_values("device_model")
y_enc = pd.factorize(y, sort=True)[0]

mi = mutual_info_classif(
    X_enc,
    y_enc,
    discrete_features=True,
    random_state=0
)

mi_scores = pd.Series(mi, index=X_raw.columns).sort_values(ascending=False)
mi_scores.plot(kind="bar")

In [ ]:
top = mi_scores.head(40).index

X_str[top].T

In [ ]:
varying = X_str.nunique(dropna=False) > 1

X_raw2 = X_raw.loc[:, varying]
X_str2 = X_str.loc[:, varying]

In [ ]:
X_enc2 = X_str2.apply(lambda s: pd.factorize(s, sort=True)[0]).to_numpy()

# y = caps_pivot.index.get_level_values("device_model")
# y_enc = pd.factorize(y, sort=True)[0]

mi2 = mutual_info_classif(
    X_enc2,
    y_enc,
    discrete_features=True,
    random_state=0
)

mi_scores2 = pd.Series(mi2, index=X_raw2.columns).sort_values(ascending=False)
mi_scores2

In [ ]:
mi_scores2.plot(kind="bar")

In [ ]:
top = mi_scores2.head(40).index

print(X_str2[top].T)

In [ ]:
X_top = X_raw2[top]
X_top.groupby(list(X_top.columns), dropna=False).apply(lambda df: df.index.tolist()).tolist()

In [ ]:


def find_pure_rules(X, y):
    rules = []
    for col in X.columns:
        groups = X.groupby(col).groups
        for val, idx_labels in groups.items():
            labels_in_group = y.loc[idx_labels]
            counts = Counter(labels_in_group.tolist())
            if len(counts) == 1:
                predicted_type = next(iter(counts.keys()))
                coverage = len(labels_in_group)
                
                rules.append({
                    "flag": col,
                    "value": val,
                    "device_type": predicted_type,
                    "coverage": coverage,
                })
    return sorted(rules, key=lambda r: (-r["coverage"], r["flag"]))

In [ ]:
dl_X = caps_pivot.astype("string").fillna("<<MISSING>>")
dl_X

In [ ]:
dl_y = pd.Series(caps_pivot.index.get_level_values("device_type"), index=caps_pivot.index, name="device_type")
dl_y

In [ ]:
find_pure_rules(dl_X, dl_y)

In [ ]:
def greedy_decision_list(X, y):
    remaining = X.index
    decision_list = []

    while len(set(y.loc[remaining])) > 1:
        Xr = X.loc[remaining]
        yr = y.loc[remaining]

        rules = find_pure_rules(Xr, yr)
        
        if not rules:
            break

        best = rules[0]
        flag, value, dt = best["flag"], best["value"], best["device_type"]

        covered = Xr.index[Xr[flag] == value]
        decision_list.append((flag, value, dt, len(covered)))

        remaining = remaining.difference(covered)

    # -- stall detection --
    remaining_types = set(y.loc[remaining]) if len(remaining) else set()
    stalled = (len(remaining) > 0) and (len(remaining_types) > 1)

    # -- default rule --
    if len(remaining) > 0:
        default_type = y.loc[remaining].mode().iloc[0]
        default_coverage = len(remaining)
        default_correct = int((y.loc[remaining] == default_type).sum())
        default_accuracy = default_correct / default_coverage
    else:
        default_type = y.mode().iloc[0]
        default_coverage = 0
        default_correct = 0
        default_accuracy = None
    # default = y.loc[remaining].mode().iloc[0] if len(remaining) else y.mode().iloc[0]

    diagnostics = {
        "stalled": stalled,
        "remaining_samples": len(remaining),
        "remaining_types": remaining_types,
        "default_coverage": default_coverage,
        "default_accuracy_on_remaining": default_accuracy,
    }
    
    return decision_list, default_type, diagnostics

In [ ]:
dl, default, diag = greedy_decision_list(dl_X, dl_y)

dl, default, diag

In [ ]:
for i, (flag, value, dt, cov) in enumerate(dl, 1):
    print(f"{"ELSE " if i > 1 else ""}IF {flag} == {value} -> {dt} (covers {cov})")
print(f"ELSE -> {default}")

In [ ]:
def greedy_decision_list_2(X, y):
    remaining = X.index
    decision_list = []

    while len(set(y.loc[remaining])) > 1:
        Xr = X.loc[remaining]
        yr = y.loc[remaining]

        rules = find_pure_rules(Xr, yr)
        
        if not rules:
            break

        best = rules[0]
        flag, value, dt = best["flag"], best["value"], best["device_type"]

        covered = Xr.index[Xr[flag] == value]
        decision_list.append((flag, value, dt, len(covered), rules))

        remaining = remaining.difference(covered)

    # -- stall detection --
    remaining_types = set(y.loc[remaining]) if len(remaining) else set()
    stalled = (len(remaining) > 0) and (len(remaining_types) > 1)

    # -- default rule --
    if len(remaining) > 0:
        default_type = y.loc[remaining].mode().iloc[0]
        default_coverage = len(remaining)
        default_correct = int((y.loc[remaining] == default_type).sum())
        default_accuracy = default_correct / default_coverage
    else:
        default_type = y.mode().iloc[0]
        default_coverage = 0
        default_correct = 0
        default_accuracy = None
    # default = y.loc[remaining].mode().iloc[0] if len(remaining) else y.mode().iloc[0]

    diagnostics = {
        "stalled": stalled,
        "remaining_samples": len(remaining),
        "remaining_types": remaining_types,
        "default_coverage": default_coverage,
        "default_accuracy_on_remaining": default_accuracy,
    }
    
    return decision_list, default_type, diagnostics

In [ ]:
dl2, default2, _ = greedy_decision_list_2(dl_X, dl_y)

In [ ]:
def format_cond(flag, value, cov, neg=False):
    op = "!=" if neg else "=="
    return f"{flag} {op} {value} ({cov})"

def format_conjunction(conds):
    return " AND ".join(conds) if conds else "<TRUE>"

def print_conjunction_rules(decision_list):
    prior_neg = []

    for step, (chosen_flag, chosen_value, chosen_dt, cov, rules) in enumerate(decision_list, 1):
        print(f"Step {step}")
        print(f"Chosen rule: IF {chosen_flag} == {chosen_value} -> {chosen_dt}")

        print("Alternatives")

        for r in rules:
            conds = []

            conds.append(format_cond(r["flag"], r["value"], r["coverage"]))

            conj = format_conjunction(conds)

            print(f"   IF {conj} -> {r['device_type']}")

In [ ]:
print_conjunction_rules(dl2)